In [1]:
from ase.io import read
from ase.visualize import view

atoms = read("71_7114_7_heur18.traj", index=":")
view(atoms)  # 打开一个交互窗口

Traceback (most recent call last):
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/site-packages/ase/__main__.py", line 2, in <module>
    main()
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/site-packages/ase/cli/main.py", line 64, in main
    cmd = import_module(module_name).CLICommand
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "<frozen importlib._bootstrap>", line 1050, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1027, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1006, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 

BrokenPipeError: [Errno 32] Broken pipe

In [1]:
from ase.io import Trajectory

# 打开轨迹文件
traj = Trajectory("/root/autodl-tmp/AdsorbDiff/0/relaxations/12_1174_85.traj")

# 看看一共多少帧
print("总帧数:", len(traj))

# 随便拿第一帧
atoms = traj[4]

# ---- 1. atoms 本身的信息 ----
print("原子数:", len(atoms))
print("化学式:", atoms.get_chemical_formula())
print("原子坐标:", atoms.positions)
print("晶胞:", atoms.cell)

# ---- 2. 附加字典 info ----
print("info 里的键:", atoms.info.keys())
print("info 内容:", atoms.info)

# ---- 3. 每个原子附加数组 ----
print("arrays 里的键:", atoms.arrays.keys())
for k, v in atoms.arrays.items():
    print(f"{k}: shape {v.shape}")

# ---- 4. 如果有计算结果（DFT/MLFF） ----
if atoms.calc is not None:
    print("能量:", atoms.get_potential_energy())
    print("力:", atoms.get_forces())


总帧数: 300
原子数: 64
化学式: CH3S36Sn24
原子坐标: [[ 3.06547284e+00  6.55822992e+00  1.53189421e+01]
 [ 6.39808989e+00  8.05874634e+00  1.73967094e+01]
 [ 7.94339848e+00  7.42541194e-01  2.30421410e+01]
 [ 1.68448746e+00  1.37390270e+01  1.36220522e+01]
 [ 9.21408463e+00  9.51594067e+00  1.54836884e+01]
 [-1.07328391e+00  5.04688311e+00  1.67951508e+01]
 [ 5.69622278e+00  1.13224802e+01  1.53690653e+01]
 [ 4.64872551e+00  1.77705944e+00  2.09373779e+01]
 [ 3.18919277e+00  6.54072905e+00  1.14601879e+01]
 [ 6.14405537e+00  7.99035406e+00  1.33879032e+01]
 [ 7.88199854e+00  7.16628909e-01  1.91412106e+01]
 [ 1.48788476e+00  1.38277159e+01  9.53827477e+00]
 [ 9.10914898e+00  9.62771034e+00  1.14758024e+01]
 [ 2.77517885e-01  4.92724991e+00  1.33904610e+01]
 [ 4.38671160e+00  1.21833897e+01  1.14964762e+01]
 [ 4.92923164e+00  2.21185184e+00  1.70653439e+01]
 [ 3.12825084e+00  6.58463192e+00  7.65925837e+00]
 [ 6.04142857e+00  7.99654388e+00  9.59772205e+00]
 [ 7.89772749e+00  7.07909346e-01  1.525306

In [4]:
import pickle
from ase.io import read
import numpy as np

# 载入 tags
with open("oc20dense_tags.pkl", "rb") as f:
    tags_dict = pickle.load(f)

# 选一个系统，例如 '71_7114_7'
system_id = "71_7114_7"
traj_file = f"{system_id}_heur18.traj"   # 这里换成你实际的 traj 文件路径

# 读取 traj（取第一帧）
atoms = read(traj_file, index=0)

# 获取对应的标签
tags = tags_dict[system_id]   # numpy 数组，长度等于原子数

# 分别取 slab / adsorbate
slab_idx = np.where(tags == 0)[0]
ads_idx = np.where((tags == 1) | (tags == 2))[0]
ads_center_idx = np.where(tags == 2)[0]

print(f"总原子数: {len(tags)}")
print(f"Slab 原子数: {len(slab_idx)}")
print(f"Adsorbate 原子数: {len(ads_idx)} (其中锚点 {len(ads_center_idx)})")

# 提取 adsorbate 的坐标
ads_coords = atoms.positions[ads_idx]
print("Adsorbate 坐标:")
print(ads_coords)

# 示例：计算 adsorbate 的质心
ads_com = ads_coords.mean(axis=0)
print("Adsorbate 质心:", ads_com)

# 示例：计算旋转矩阵（这里随便举个例子，把 adsorbate 对齐到 z 轴）
def compute_rotation_matrix(vec, target=np.array([0,0,1])):
    vec = vec / np.linalg.norm(vec)
    target = target / np.linalg.norm(target)
    v = np.cross(vec, target)
    c = np.dot(vec, target)
    if np.allclose(v, 0):
        return np.eye(3)
    vx = np.array([[0, -v[2], v[1]],
                   [v[2], 0, -v[0]],
                   [-v[1], v[0], 0]])
    R = np.eye(3) + vx + vx @ vx * ((1-c)/(np.linalg.norm(v)**2))
    return R

# 举例：adsorbate 第一个原子相对质心的方向
vec = ads_coords[0] - ads_com
R = compute_rotation_matrix(vec)
print("旋转矩阵 R:")
print(R)

# 应用旋转矩阵
ads_rotated = (ads_coords - ads_com) @ R.T + ads_com
print("旋转后的第一个原子:", ads_rotated[0])


总原子数: 67
Slab 原子数: 48
Adsorbate 原子数: 19 (其中锚点 3)
Adsorbate 坐标:
[[ 2.10054599  2.72280721 23.50017375]
 [ 2.10054599  7.84809136 23.04715835]
 [ 0.          9.44974266 21.68811215]
 [ 4.20109197  4.32445851 22.14112755]
 [ 6.30163796  2.72280721 23.50017375]
 [ 6.30163796  7.84809136 23.04715835]
 [ 4.20109197  9.44974266 21.68811215]
 [ 0.          4.32445851 22.14112755]
 [ 2.10054599  5.28544928 23.27366605]
 [ 4.20109197  1.76181643 22.36763525]
 [ 6.30163796  5.28544928 23.27366605]
 [ 0.          1.76181643 22.36763525]
 [ 0.          6.88710058 21.91461985]
 [ 2.10054599  0.16016513 23.72668145]
 [ 4.20109197  6.88710058 21.91461985]
 [ 6.30163796  0.16016513 23.72668145]
 [ 4.45516477  5.76304735 23.4627967 ]
 [ 4.6920846   6.68252323 23.83539992]
 [ 4.72824499  4.92134582 23.97010892]]
Adsorbate 质心: [ 3.38361042  4.9603252  22.87298181]
旋转矩阵 R:
[[ 0.81101347 -0.32957095  0.48336337]
 [-0.32957095  0.42526586  0.84293056]
 [-0.48336337 -0.84293056  0.23627933]]
旋转后的第一个原子: [ 3.38

In [8]:
import pickle, os, re
import numpy as np
from ase.io import Trajectory

# ---- 读 tags.pkl ----
with open("oc20dense_tags.pkl", "rb") as f:
    tags_map = pickle.load(f)

# ---- 从文件名推 sid 的一个例子（按你的命名规则调整）----
traj_path = "71_7114_7_heur18.traj"

sid = "71_7114_7"

# ---- 取 adsorbate 索引 ----
tags = tags_map[sid]   # numpy array, shape=(natoms,)
ads_idx = np.where((tags == 1) | (tags == 2))[0]  # 1/2 都当作 adsorbate
print(tags)
# ---- 读初末帧 ----
traj = Trajectory(traj_path)
ai, af = traj[0], traj[-1]
P = ai.positions[ads_idx]   # (N,3)
Q = af.positions[ads_idx]   # (N,3)

# ---- Kabsch ----
def kabsch(P, Q):
    comP, comQ = P.mean(axis=0), Q.mean(axis=0)
    P0, Q0 = P - comP, Q - comQ
    C = P0.T @ Q0
    U, S, Vt = np.linalg.svd(C)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = comQ - R @ comP
    return R, t, comP, comQ

R, t, comP, comQ = kabsch(P, Q)

# ---- 验证 ----
def rmsd(A, B):
    return float(np.sqrt(np.mean(np.sum((A - B)**2, axis=1))))

P_aligned = (R @ P.T).T + t
rmsd_val = rmsd(P_aligned, Q)

orth_err = np.linalg.norm(R.T @ R - np.eye(3), ord='fro')
detR = np.linalg.det(R)
com_map_err = np.linalg.norm(t - (comQ - R @ comP))

# 内部距离误差
def pairwise_dists(X):
    diff = X[:,None,:] - X[None,:,:]
    return np.sqrt((diff**2).sum(-1))
dP = pairwise_dists(P)
dQ = pairwise_dists(Q)
dist_err = np.mean(np.abs(dP - dQ))

print(f"RMSD (Å): {rmsd_val:.4f}")
print(f"||R^T R - I||_F: {orth_err:.2e}")
print(f"det(R): {detR:.6f}")
print(f"|t - (COM_f - R*COM_i)|: {com_map_err:.3e}")
print(f"mean |pairwise dist diff| (Å): {dist_err:.4e}")


[0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 0 0 1 0 0 0 0 1 1 0 0 0 0
 0 0 1 1 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 1 2 2 2]
RMSD (Å): 0.4467
||R^T R - I||_F: 5.65e-16
det(R): 1.000000
|t - (COM_f - R*COM_i)|: 0.000e+00
mean |pairwise dist diff| (Å): 1.6662e-01


In [6]:

import os
import json
import pickle
import numpy as np
from typing import Tuple, Optional

from ase.io import read
from ase.geometry import cell_to_cellpar

# ===================== CONFIG (edit me) =====================

TRAJ_PATH = "71_7114_7_heur18.traj"     # your .traj file
TAGS_PKL  = "oc20dense_tags.pkl"        # your tags map pkl
SID       = "71_7114_7"                 # key in the tags map
T_SAMPLE  = 0.5                          # time in [0,1] to sample

# If you know which tag values correspond to the adsorbate, set them here:
# e.g., (1,) or (1,2). If None, will auto-pick the minority label as adsorbate.
ADSORBATE_TAG_VALUES: Optional[Tuple[int, ...]] = None

# Minimum adsorbate atoms to accept
KMIN_ADS = 2

# Output file
OUT_NPZ = "flowmatch_sample_hardcoded.npz"

# ===================== Utilities =====================

def wrap_frac_delta(d):
    return (d + 0.5) % 1.0 - 0.5

def kabsch(P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    H = P.T @ Q
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1.0
        R = Vt.T @ U.T
    return R

def log_SO3(R: np.ndarray) -> np.ndarray:
    tr = np.clip(np.trace(R), -1.0, 3.0)
    theta = np.arccos((tr - 1.0) / 2.0)
    if theta < 1e-8:
        return np.zeros((3,3))
    skew = (R - R.T) / (2.0 * np.sin(theta))
    return skew * theta

def exp_SO3(omega: np.ndarray) -> np.ndarray:
    theta = np.linalg.norm([omega[2,1], omega[0,2], omega[1,0]])
    if theta < 1e-8:
        return np.eye(3)
    K = omega / theta
    return np.eye(3) + np.sin(theta)*K + (1.0 - np.cos(theta))*(K @ K)

def segment_by_tags(tags: np.ndarray,
                    adsorbate_tag_values: Optional[Tuple[int, ...]],
                    kmin_ads: int=2) -> np.ndarray:
    """Return boolean mask for adsorbate using tags.
       If adsorbate_tag_values is None, pick the minority label."""
    tags = np.asarray(tags)
    if adsorbate_tag_values is not None:
        sel = np.zeros_like(tags, dtype=bool)
        for v in adsorbate_tag_values:
            sel |= (tags == v)
    else:
        uniq, counts = np.unique(tags, return_counts=True)
        if len(uniq) == 1:
            # ambiguous; no split
            sel = np.zeros_like(tags, dtype=bool)
        else:
            sel = (tags == uniq[np.argmin(counts)])
    # sanity
    if sel.sum() < kmin_ads or sel.sum() >= len(tags):
        raise RuntimeError(f"Invalid adsorbate mask from tags (sum={sel.sum()} of {len(tags)}). "
                           f"Consider setting ADSORBATE_TAG_VALUES explicitly.")
    return sel

# ===================== Core =====================

def build_flowmatch_sample_hardcoded():
    # Load trajectory (first and last frame)
    images = read(TRAJ_PATH, index=":")
    if len(images) < 2:
        raise RuntimeError(f"Need at least two frames in {TRAJ_PATH}, got {len(images)}")
    atoms0 = images[0].copy()
    atoms1 = images[-1].copy()

    # Cell, positions
    cell = atoms0.get_cell()
    a1, a2, a3 = np.array(cell)
    cellpar = cell_to_cellpar(cell)

    X0 = atoms0.get_positions()
    X1 = atoms1.get_positions()

    # Load tags and build adsorbate mask
    if not os.path.isfile(TAGS_PKL):
        raise FileNotFoundError(f"Tags pkl not found: {TAGS_PKL}")
    with open(TAGS_PKL, "rb") as f:
        tags_map = pickle.load(f)
    if SID not in tags_map:
        raise KeyError(f"SID '{SID}' not found in tags map.")
    tags = np.asarray(tags_map[SID])
    if tags.shape[0] != X0.shape[0]:
        raise ValueError(f"Tags length ({tags.shape[0]}) != natoms ({X0.shape[0]}). Check your inputs.")

    mask_ad = segment_by_tags(tags, ADSORBATE_TAG_VALUES, kmin_ads=KMIN_ADS)

    # Extract adsorbate coordinates
    ads0 = X0[mask_ad]
    ads1 = X1[mask_ad]

    # Centers of mass
    com0 = ads0.mean(axis=0)
    com1 = ads1.mean(axis=0)

    # Fractional COM, in-plane shortest translation on T^2
    scaled0 = atoms0.get_scaled_positions(wrap=True)
    scaled1 = atoms1.get_scaled_positions(wrap=True)
    scaled_com0 = scaled0[mask_ad].mean(axis=0)
    scaled_com1 = scaled1[mask_ad].mean(axis=0)
    dfrac = np.array([wrap_frac_delta(scaled_com1[0]-scaled_com0[0]),
                      wrap_frac_delta(scaled_com1[1]-scaled_com0[1]),
                      0.0])
    delta_r_xy = dfrac[0]*a1 + dfrac[1]*a2  # (3,)

    # Kabsch rotation around COM
    P = ads0 - com0
    Q = ads1 - com1
    R_star = kabsch(P, Q)

    # Log/exp on SO(3)
    Omega = log_SO3(R_star)
    axis = np.array([Omega[2,1], Omega[0,2], Omega[1,0]])
    angle = np.linalg.norm(axis)
    if angle > 1e-12:
        axis = axis / angle
    else:
        axis = np.array([1.0, 0.0, 0.0])

    # Build trajectory and instantaneous velocity at T_SAMPLE
    t = float(T_SAMPLE)
    R_t = exp_SO3(Omega * t)
    r_t = delta_r_xy * t
    X_t = (R_t @ P.T).T + com0 + r_t
    dXdt_t = (Omega @ R_t @ P.T).T + delta_r_xy

    # Save
    np.savez(OUT_NPZ,
             X0_ads=ads0, X1_ads=ads1,
             R_star=R_star, Omega=Omega, delta_r_xy=delta_r_xy,
             t=np.array([t]), X_t=X_t, dXdt_t=dXdt_t,
             mask_adsorbate=mask_ad.astype(np.int8),
             cellpar=cellpar,
             rotation_axis=axis, rotation_angle_deg=np.degrees(angle),
             traj_path=np.array([TRAJ_PATH]), sid=np.array([SID]),
             selected_by=np.array(["tags.pkl"]))

    # Diagnostics
    info = {
        "selected_by": "tags.pkl",
        "num_atoms_total": int(len(X0)),
        "num_adsorbate_atoms": int(P.shape[0]),
        "cell_parameters_(a,b,c,alpha,beta,gamma)": [float(x) for x in cellpar],
        "rotation_angle_deg": float(np.degrees(angle)),
        "rotation_axis": [float(x) for x in axis],
        "delta_r_xy_Ang": [float(x) for x in delta_r_xy],
        "com0": [float(x) for x in com0],
        "com1": [float(x) for x in com1],
        "t": t,
        "npz_out": OUT_NPZ
    }
    print(json.dumps(info, indent=2))

if __name__ == "__main__":
    build_flowmatch_sample_hardcoded()


{
  "selected_by": "tags.pkl",
  "num_atoms_total": 67,
  "num_adsorbate_atoms": 3,
  "cell_parameters_(a,b,c,alpha,beta,gamma)": [
    8.40218394,
    10.29053169124535,
    38.95932419,
    95.05115255365733,
    90.0,
    90.0
  ],
  "rotation_angle_deg": 32.60892247757106,
  "rotation_axis": [
    -0.2305059877757972,
    -0.9654676291964586,
    0.12140529878581777
  ],
  "delta_r_xy_Ang": [
    -0.4266640207643909,
    -0.5817627698108402,
    0.05142105020925727
  ],
  "com0": [
    4.625164785244576,
    5.788972131892121,
    23.756101846800703
  ],
  "com1": [
    4.198500764480184,
    5.20720936208128,
    24.63079588192778
  ],
  "t": 0.5,
  "npz_out": "flowmatch_sample_hardcoded.npz"
}


/tmp/ipykernel_103266/1948262485.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  a1, a2, a3 = np.array(cell)


In [2]:
import numpy as np

# 修改为你的文件路径
file_path = "/root/autodl-tmp/AdsorbDiff/0/781426102.npz"

# 加载 npz 文件
data = np.load(file_path)

# 打印里面的所有键（数组名称）
print("Keys in npz file:", data.files)

# 遍历并打印每个数组的形状和部分数据
for key in data.files:
    arr = data[key]
    print(f"\nKey: {key}")
    print("Shape:", arr.shape)
    print("Data (first 10 elements):", arr.ravel()[:10])  # 展开并只显示前 5 个元素


Keys in npz file: ['sid', 'natoms', 'z', 'cell', 'pbc', 'pos_traj']

Key: sid
Shape: ()
Data (first 10 elements): [781426102]

Key: natoms
Shape: ()
Data (first 10 elements): [52]

Key: z
Shape: (52,)
Data (first 10 elements): [20. 20. 20. 20. 20. 20. 20. 20. 20. 20.]

Key: cell
Shape: (3, 3)
Data (first 10 elements): [10.4639     0.        -0.         0.        12.525262   2.4576273
  0.         0.        38.292282 ]

Key: pbc
Shape: (3,)
Data (first 10 elements): [ True  True False]

Key: pos_traj
Shape: (101, 52, 3)
Data (first 10 elements): [ 1.3079875  3.6862264 18.498096   1.3187805 10.0070505 26.426107
  3.9239624  8.839035  16.043327   3.9239624]
